# Adverse Drug Reaction (ADR) Detection from Patient-Reported Text
### NLP with Transformers (DistilBERT) — Medical Text Classification

**Author:** Aidan Agee  
**Dataset:** [SMM4H 2019 Task 1](https://healthlanguageprocessing.org/smm4h-2019/) — Social Media Mining for Health  
**Framework:** PyTorch + HuggingFace Transformers  
**Task:** Binary classification — does this text mention an adverse drug reaction?

---

## Overview

Adverse Drug Reactions (ADRs) are unintended harmful effects caused by medications. They account for over **1 million hospitalizations per year** in the US and are severely underreported through official channels.

Patients increasingly describe their medication experiences on social media and review platforms. This project fine-tunes a **DistilBERT** transformer model to automatically detect ADR mentions in patient-reported text — enabling large-scale pharmacovigilance (drug safety monitoring) that would be impossible to do manually.

This is an active area of real-world NLP research used by pharmaceutical companies and regulatory agencies like the FDA.

### This project is part of a medical AI series:
- [Intracranial Aneurysm Detection](https://github.com/aidanagee/aneurysm-detection-rsna) — 3D CV segmentation
- [Lung Nodule Detection](https://github.com/aidanagee/lung-nodule-detection-3d-unet) — 3D U-Net segmentation  
- **ADR Detection (this project)** — Medical NLP classification

### Key Concepts Covered
- Transformer architecture and BERT/DistilBERT fundamentals
- Fine-tuning pretrained language models for domain-specific classification
- Tokenization and attention masks
- Handling class imbalance in NLP tasks
- Evaluation metrics beyond accuracy (F1, precision, recall)
- Inference pipeline for new text inputs


---
## 1. Background: Why Transformers for Medical NLP?

### The Problem with Traditional NLP
Earlier approaches (bag-of-words, TF-IDF + logistic regression) treat text as unordered word counts. They miss:
- **Context:** "The drug didn't cause nausea" vs "The drug caused nausea" — opposite meanings, same key words
- **Negation:** Critical in medical text
- **Word sense:** "I felt sick after taking it" — "sick" needs surrounding context

### BERT and the Transformer Revolution
**BERT** (Bidirectional Encoder Representations from Transformers, Google 2018) changed NLP fundamentally:
- Pretrained on massive text corpora — learns general language understanding
- **Bidirectional** — reads text in both directions simultaneously, capturing full context
- Fine-tunable — the pretrained weights are adapted to specific tasks with minimal additional training data

### Why DistilBERT specifically?
**DistilBERT** is a distilled (compressed) version of BERT:

| Model | Parameters | Speed | Accuracy vs BERT |
|---|---|---|---|
| BERT-base | 110M | 1x | 100% |
| DistilBERT | 66M | 1.6x faster | ~97% |
| BERT-large | 340M | 0.5x | 105% |

For a project like this, DistilBERT gives us ~97% of BERT's performance at 40% fewer parameters — a good trade-off for training without massive GPU resources.

### Why not just use GPT/LLMs?
GPT-style models are generative — they predict the next token. For classification tasks, encoder models like BERT/DistilBERT are more efficient and often more accurate. We don't need text generation, we need a binary decision.


---
## 2. Imports and Setup

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# HuggingFace
try:
    from transformers import (
        DistilBertTokenizer,
        DistilBertForSequenceClassification,
        get_linear_schedule_with_warmup
    )
    print("HuggingFace Transformers available")
except ImportError:
    print("Install with: pip install transformers")

try:
    from sklearn.metrics import (
        classification_report, confusion_matrix,
        f1_score, precision_score, recall_score
    )
    print("Scikit-learn available")
except ImportError:
    print("Install with: pip install scikit-learn")

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
print(f"PyTorch: {torch.__version__}")

---
## 3. Dataset: SMM4H 2019 + Synthetic Augmentation

### About SMM4H
The Social Media Mining for Health (SMM4H) shared task provides annotated tweets mentioning medications. Task 1 is binary classification: does this tweet mention an ADR?

**Dataset characteristics:**
- ~25,000 annotated tweets
- Class imbalance: ~85% non-ADR, ~15% ADR (real ADR mentions are rare)
- Noisy social media text: abbreviations, misspellings, slang

### Examples:
| Text | Label |
|---|---|
| "Been on Zoloft for 3 weeks, insomnia is unbearable" | ADR ✓ |
| "Tylenol is great for headaches" | Not ADR ✗ |
| "metformin making me so nauseous I can barely eat" | ADR ✓ |
| "picked up my prescription today" | Not ADR ✗ |

We include synthetic examples representative of the dataset style for demonstration.


In [ ]:
def get_synthetic_dataset():
    """
    Synthetic dataset representative of SMM4H ADR detection task.
    
    In production: replace with real SMM4H data from:
    https://healthlanguageprocessing.org/smm4h-2019/
    
    Label: 1 = ADR mentioned, 0 = no ADR
    """
    adr_examples = [
        # Clear ADRs
        ("Been on Zoloft for 3 weeks and the insomnia is absolutely unbearable", 1),
        ("metformin is making me so nauseous I can barely eat anything", 1),
        ("prednisone has me gaining weight so fast it's ridiculous", 1),
        ("lisinopril gave me the worst dry cough I've ever had", 1),
        ("lexapro side effects are no joke, constant headaches and dizziness", 1),
        ("accutane destroyed my lips they're cracking and bleeding every day", 1),
        ("been on abilify 2 weeks and I can't stop moving my legs at night", 1),
        ("humira injection sites are so itchy and red, is this normal?", 1),
        ("amoxicillin gave me a horrible rash all over my arms", 1),
        ("adderall is making my heart race so fast I had to call my doctor", 1),
        ("can't sleep on this medication, been awake for 2 days straight", 1),
        ("methotrexate making my hair fall out in clumps, so scared", 1),
        ("cymbalta withdrawal is the worst thing I've ever experienced", 1),
        ("stopped taking lipitor because the muscle pain was unbearable", 1),
        ("my anxiety got so much worse after starting this medication", 1),
        ("prozac is causing extreme sweating, so embarrassing", 1),
        ("warfarin bruising is insane, bumped my arm and it went black", 1),
        ("birth control pill giving me daily migraines for 3 months now", 1),
        ("sertraline making me feel completely numb and emotionless", 1),
        ("had an allergic reaction to penicillin, throat swelling up", 1),
        ("quetiapine weight gain is real, up 20 pounds in 2 months", 1),
        ("this antibiotic is giving me severe stomach cramps", 1),
        ("beta blockers making me so tired I can barely function at work", 1),
        ("started wellbutrin and now I have constant ringing in my ears", 1),
        ("ibuprofen long term use gave me a stomach ulcer", 1),
        # Subtle ADRs
        ("anyone else feel foggy on this medication? can't concentrate at all", 1),
        ("day 5 on new meds and I feel worse than before I started", 1),
        ("my vision has been blurry since I started the new prescription", 1),
        ("feel so shaky and weak since upping my dose", 1),
        ("skin peeling on my hands since starting the new treatment", 1),
        # Non-ADR
        ("just picked up my prescription from the pharmacy", 0),
        ("tylenol is great for my headaches", 0),
        ("reminder to take your medication every day at the same time", 0),
        ("my doctor prescribed me metformin for diabetes management", 0),
        ("does anyone else take vitamin D supplements in winter?", 0),
        ("finally got my insurance to cover my medication", 0),
        ("asking for a friend — is it safe to take ibuprofen with food?", 0),
        ("new study shows aspirin may reduce heart disease risk", 0),
        ("pharmacist was so helpful explaining my new prescription", 0),
        ("taking my antibiotics as prescribed, hoping this infection clears up", 0),
        ("anyone have recommendations for a good probiotic?", 0),
        ("my doctor increased my dose today, fingers crossed it helps", 0),
        ("been on this medication for years and it works great for me", 0),
        ("picked up my son's allergy medication today", 0),
        ("grateful for modern medicine honestly", 0),
        ("is it okay to take melatonin occasionally for sleep?", 0),
        ("asked my doctor about switching medications today", 0),
        ("generic vs brand name — is there actually a difference?", 0),
        ("flu shot done for the year!", 0),
        ("researching side effects before starting new medication", 0),
        ("medication reminder apps are actually super helpful", 0),
        ("hope this prescription helps, been struggling for months", 0),
        ("drug interactions checker online is really useful", 0),
        ("finally found a medication that works for my condition", 0),
        ("second opinion confirmed my current treatment plan is right", 0),
    ]
    
    # Expand dataset with variations for training robustness
    expanded = []
    for text, label in adr_examples:
        expanded.append((text, label))
        # Add capitalization variation
        expanded.append((text.upper(), label))
        # Add with common social media abbreviations
        if label == 1:
            expanded.append((text.replace("medication", "meds").replace("prescription", "rx"), label))
    
    random.shuffle(expanded)
    return expanded


data = get_synthetic_dataset()
labels = [d[1] for d in data]
print(f"Total samples: {len(data)}")
print(f"ADR (positive): {sum(labels)} ({100*sum(labels)/len(labels):.1f}%)")
print(f"Non-ADR (negative): {len(labels)-sum(labels)} ({100*(len(labels)-sum(labels))/len(labels):.1f}%)")
print(f"\nClass imbalance ratio: {(len(labels)-sum(labels))/sum(labels):.1f}:1")
print("\nSample ADR examples:")
for text, label in [d for d in data if d[1]==1][:3]:
    print(f"  [ADR] {text[:80]}")
print("\nSample non-ADR examples:")
for text, label in [d for d in data if d[1]==0][:3]:
    print(f"  [NEG] {text[:80]}")

---
## 4. Tokenization: How BERT Reads Text

Transformers don't read raw text — they read **tokens**, which are subword units from a fixed vocabulary. Understanding tokenization is critical for using BERT correctly.

### Key concepts:

**[CLS] token:** A special token prepended to every sequence. Its final hidden state is used as the sequence representation for classification tasks. Think of it as "the model's summary of the entire input."

**[SEP] token:** Marks the end of a sequence (or separation between two sequences in tasks like question answering).

**Attention mask:** Tells the model which tokens are real content (1) vs padding (0). When we batch sequences of different lengths, shorter sequences are padded to match the longest — the attention mask prevents the model from "attending to" meaningless padding tokens.

**Max length:** We truncate/pad all sequences to 128 tokens. Most tweets are well under this limit.

### Example tokenization:
```
Input: "metformin is making me nauseous"
Tokens: [CLS] met ##for ##min is making me nauseous [SEP] [PAD] [PAD]...
IDs:    [101] [4  ][5  ][6  ] [7] [8    ] [9] [10    ] [102][0  ][0  ]...
Mask:   [ 1 ] [1  ][1  ][1  ] [1] [1    ] [1] [1     ] [1  ][0  ][0  ]...
```
Note how "metformin" is split into subword pieces (met + ##for + ##min) — this lets BERT handle rare medical drug names it may not have seen during pretraining.


In [ ]:
# Load DistilBERT tokenizer
# This downloads ~250MB of pretrained weights on first run
print("Loading DistilBERT tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
print(f"Vocabulary size: {tokenizer.vocab_size:,} tokens")

# Demonstrate tokenization
example_texts = [
    "metformin is making me so nauseous",
    "just picked up my prescription",
    "lisinopril gave me an awful dry cough that won't stop"
]

print("\n" + "="*60)
print("TOKENIZATION EXAMPLES")
print("="*60)

for text in example_texts:
    tokens = tokenizer.tokenize(text)
    encoding = tokenizer(text, max_length=32, padding='max_length', 
                        truncation=True, return_tensors='pt')
    print(f"\nText: '{text}'")
    print(f"Tokens ({len(tokens)}): {tokens}")
    print(f"Input IDs shape: {encoding['input_ids'].shape}")
    print(f"Attention mask sum: {encoding['attention_mask'].sum().item()} real tokens")


class ADRDataset(Dataset):
    """
    PyTorch Dataset for ADR detection.
    
    Handles tokenization and encoding of raw text for DistilBERT input.
    Each item returns:
        input_ids: token indices (B, max_len)
        attention_mask: real vs padding tokens (B, max_len)  
        label: 0 or 1
    """
    
    def __init__(self, samples, tokenizer, max_length=128):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        text, label = self.samples[idx]
        
        # Clean text — remove URLs, normalize whitespace
        text = re.sub(r'http\S+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',    # pad shorter sequences
            truncation=True,         # truncate longer sequences
            return_tensors='pt'      # return PyTorch tensors
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }


# Train/val/test split (70/15/15)
random.shuffle(data)
n = len(data)
train_data = data[:int(0.7 * n)]
val_data = data[int(0.7 * n):int(0.85 * n)]
test_data = data[int(0.85 * n):]

train_dataset = ADRDataset(train_data, tokenizer)
val_dataset = ADRDataset(val_data, tokenizer)
test_dataset = ADRDataset(test_data, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"\nDataset splits:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val:   {len(val_dataset)} samples")
print(f"  Test:  {len(test_dataset)} samples")

# Inspect a batch
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  input_ids:      {batch['input_ids'].shape}")
print(f"  attention_mask: {batch['attention_mask'].shape}")
print(f"  labels:         {batch['label'].shape} — values: {batch['label'].tolist()}")

---
## 5. Model: Fine-tuning DistilBERT

### What is fine-tuning?
DistilBERT was pretrained on Wikipedia + BooksCorpus — it has broad language understanding but no specific knowledge of ADRs or medical text.

**Fine-tuning** adapts the pretrained weights to our specific task by:
1. Adding a classification head (linear layer) on top of the [CLS] token output
2. Training the entire model end-to-end on our labeled data
3. Using a small learning rate to avoid "catastrophic forgetting" of pretrained knowledge

This is the core paradigm of modern NLP: **pretrain on large data → fine-tune on small task-specific data.**

### Why this works with limited data
A fully pretrained BERT already understands:
- "nauseous", "rash", "side effect", "reaction" are medically relevant
- Negation ("didn't cause", "no side effects")
- Context and word relationships

We just need to teach it the specific ADR classification boundary — which requires far less data than training from scratch.


In [ ]:
print("Loading DistilBERT for sequence classification...")
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2  # binary: ADR or not
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model.config)

# Test forward pass
sample_batch = next(iter(train_loader))
with torch.no_grad():
    outputs = model(
        input_ids=sample_batch['input_ids'].to(device),
        attention_mask=sample_batch['attention_mask'].to(device),
        labels=sample_batch['label'].to(device)
    )

print(f"\nForward pass output:")
print(f"  Loss: {outputs.loss.item():.4f}")
print(f"  Logits shape: {outputs.logits.shape}  (batch_size x num_labels)")
print(f"  Sample logits: {outputs.logits[0].tolist()}")
probs = torch.softmax(outputs.logits[0], dim=-1)
print(f"  Probabilities: [non-ADR: {probs[0]:.3f}, ADR: {probs[1]:.3f}]")

---
## 6. Handling Class Imbalance

ADR mentions are rare — in the real SMM4H dataset, only ~10-15% of posts mention ADRs. If we ignore this:
- Model learns to predict "non-ADR" for everything
- Gets ~85% accuracy but 0% recall on actual ADRs — completely useless clinically

### Two strategies we use:

**1. Weighted Cross Entropy Loss**  
Give higher loss penalty for misclassifying the minority class (ADR). If ADRs are 15% of data, the ADR class gets weight ~5.7x the non-ADR class.

**2. F1 Score as primary metric (not accuracy)**  
F1 = harmonic mean of precision and recall. For imbalanced medical tasks, F1 tells us if the model is actually finding ADRs, not just predicting the majority class.


In [ ]:
# Calculate class weights from training data
train_labels = [d[1] for d in train_data]
class_counts = Counter(train_labels)
total = len(train_labels)

print(f"Training set class distribution:")
print(f"  Non-ADR (0): {class_counts[0]} ({100*class_counts[0]/total:.1f}%)")
print(f"  ADR     (1): {class_counts[1]} ({100*class_counts[1]/total:.1f}%)")

# Weight = total / (n_classes * class_count)
weight_0 = total / (2 * class_counts[0])
weight_1 = total / (2 * class_counts[1])
class_weights = torch.tensor([weight_0, weight_1]).to(device)
print(f"\nClass weights: [non-ADR: {weight_0:.3f}, ADR: {weight_1:.3f}]")
print(f"ADR misclassification penalized {weight_1/weight_0:.1f}x more than non-ADR")

criterion = nn.CrossEntropyLoss(weight=class_weights)


def compute_metrics(all_labels, all_preds):
    """
    Compute classification metrics.
    
    For medical ADR detection, we care most about:
    - Recall (sensitivity): Are we catching actual ADRs? Missing an ADR is dangerous.
    - Precision: When we say ADR, are we right? Too many false alarms reduces trust.
    - F1: Balance of both.
    """
    accuracy = np.mean(np.array(all_labels) == np.array(all_preds))
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

---
## 7. Training with Linear Warmup Scheduler

### Learning rate warmup
A key technique for fine-tuning transformers: start with a very small LR, gradually increase to the target LR over the first ~10% of training steps, then linearly decay to 0.

**Why warmup?** The classification head is randomly initialized. Early in training, large gradients from the untrained head could destabilize the pretrained BERT weights. Warmup gives the head time to stabilize before applying the full learning rate.


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    
    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        
        loss.backward()
        
        # Gradient clipping — important for transformer fine-tuning
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    metrics = compute_metrics(all_labels, all_preds)
    metrics['loss'] = total_loss / len(loader)
    return metrics


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    metrics = compute_metrics(all_labels, all_preds)
    metrics['loss'] = total_loss / len(loader)
    return metrics, all_labels, all_preds


# Training setup
N_EPOCHS = 4
LR = 2e-5  # Standard fine-tuning LR for BERT models
WARMUP_RATIO = 0.1

total_steps = len(train_loader) * N_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Training configuration:")
print(f"  Epochs: {N_EPOCHS}")
print(f"  Learning rate: {LR}")
print(f"  Total steps: {total_steps}")
print(f"  Warmup steps: {warmup_steps} ({100*WARMUP_RATIO:.0f}% of training)")
print()

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': [],
           'train_recall': [], 'val_recall': []}

best_val_f1 = 0.0
best_state = None

print(f"{'Epoch':>6} {'Tr Loss':>9} {'Val Loss':>9} {'Tr F1':>8} {'Val F1':>8} {'Val Recall':>12} {'Val Prec':>10}")
print("-" * 70)

for epoch in range(1, N_EPOCHS + 1):
    train_metrics = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_metrics, val_labels, val_preds = evaluate(model, val_loader, criterion, device)
    
    for key in ['loss', 'f1', 'recall']:
        history[f'train_{key}' if key != 'loss' else 'train_loss'].append(
            train_metrics[key if key != 'loss' else 'loss'])
        history[f'val_{key}' if key != 'loss' else 'val_loss'].append(
            val_metrics[key if key != 'loss' else 'loss'])
    
    history['train_f1'].append(train_metrics['f1'])
    history['val_f1'].append(val_metrics['f1'])
    history['train_recall'].append(train_metrics['recall'])
    history['val_recall'].append(val_metrics['recall'])
    
    print(f"{epoch:>6} {train_metrics['loss']:>9.4f} {val_metrics['loss']:>9.4f} "
          f"{train_metrics['f1']:>8.4f} {val_metrics['f1']:>8.4f} "
          f"{val_metrics['recall']:>12.4f} {val_metrics['precision']:>10.4f}")
    
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

if best_state:
    model.load_state_dict(best_state)
    print(f"\nRestored best model (val F1: {best_val_f1:.4f})")

---
## 8. Evaluation & Analysis

In [ ]:
# Final evaluation on test set
test_metrics, test_labels, test_preds = evaluate(model, test_loader, criterion, device)

print("=" * 50)
print("FINAL TEST SET RESULTS")
print("=" * 50)
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"F1 Score:  {test_metrics['f1']:.4f}  ← primary metric")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print()
print("Classification Report:")
print(classification_report(test_labels, test_preds, 
                             target_names=['Non-ADR', 'ADR']))


def plot_results(history, test_labels, test_preds):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss curves
    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train', markersize=5)
    axes[0].plot(epochs, history['val_loss'], 'r-o', label='Val', markersize=5)
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # F1 curves
    axes[1].plot(epochs, history['train_f1'], 'b-o', label='Train F1', markersize=5)
    axes[1].plot(epochs, history['val_f1'], 'r-o', label='Val F1', markersize=5)
    axes[1].plot(epochs, history['val_recall'], 'g--s', label='Val Recall', markersize=5)
    axes[1].set_title('F1 & Recall')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Confusion matrix
    cm = confusion_matrix(test_labels, test_preds)
    im = axes[2].imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    axes[2].set_title('Confusion Matrix (Test Set)')
    axes[2].set_xlabel('Predicted')
    axes[2].set_ylabel('Actual')
    axes[2].set_xticks([0, 1])
    axes[2].set_yticks([0, 1])
    axes[2].set_xticklabels(['Non-ADR', 'ADR'])
    axes[2].set_yticklabels(['Non-ADR', 'ADR'])
    
    for i in range(2):
        for j in range(2):
            axes[2].text(j, i, str(cm[i, j]), ha='center', va='center',
                        fontsize=14, color='white' if cm[i, j] > cm.max()/2 else 'black')
    
    plt.suptitle('DistilBERT ADR Detection — Training Results', fontsize=13)
    plt.tight_layout()
    plt.savefig('adr_results.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Saved: adr_results.png")

plot_results(history, test_labels, test_preds)

---
## 9. Inference Pipeline

A practical inference function for classifying new text — this is what a production deployment would use.


In [ ]:
def predict_adr(text, model, tokenizer, device, threshold=0.5):
    """
    Predict whether a text mentions an adverse drug reaction.
    
    Returns:
        label: 'ADR' or 'Non-ADR'
        confidence: probability of predicted class
        adr_probability: raw probability of ADR class
    """
    model.eval()
    
    # Clean and tokenize
    text_clean = re.sub(r'http\S+', '', text)
    text_clean = re.sub(r'\s+', ' ', text_clean).strip()
    
    encoding = tokenizer(
        text_clean,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    with torch.no_grad():
        outputs = model(
            input_ids=encoding['input_ids'].to(device),
            attention_mask=encoding['attention_mask'].to(device)
        )
    
    probs = torch.softmax(outputs.logits, dim=1)[0]
    adr_prob = probs[1].item()
    
    label = 'ADR' if adr_prob >= threshold else 'Non-ADR'
    confidence = adr_prob if label == 'ADR' else probs[0].item()
    
    return {
        'text': text,
        'label': label,
        'confidence': confidence,
        'adr_probability': adr_prob
    }


# Test inference on new examples
test_cases = [
    "been on this medication for a week and I have terrible nausea",
    "just got my prescription refilled at the pharmacy",
    "zoloft is giving me constant headaches and I can't sleep",
    "my doctor says this medication is working well for me",
    "horrible rash on my arms since starting the new antibiotic",
    "picked up my son's allergy meds today",
    "can't stop sweating since starting these new pills",
    "anyone know a good pharmacy app for reminders?",
]

print("INFERENCE RESULTS")
print("=" * 70)
for text in test_cases:
    result = predict_adr(text, model, tokenizer, device)
    marker = "✓ ADR" if result['label'] == 'ADR' else "  ---"
    print(f"[{marker}] ({result['adr_probability']:.2f}) {text[:65]}")

---
## 10. Discussion, Limitations & Real-World Considerations

### What this model does well
- Captures contextual clues that keyword matching would miss ("feel worse since starting" vs "starting to feel better")
- Handles informal language, abbreviations, and social media style text
- Fine-tuned efficiently with limited labeled data thanks to BERT pretraining

### Limitations
1. **Synthetic training data** — real deployment requires the full annotated SMM4H dataset
2. **Drug name coverage** — rare or newly approved drugs may not appear in pretraining data
3. **Negation** — "no side effects" correctly handled by BERT but remains a challenge for edge cases
4. **Causality vs mention** — "I take metformin for diabetes and I have nausea" doesn't mean metformin caused the nausea
5. **Severity** — model detects presence of ADR but doesn't classify severity

### Real-World Extensions
- **Named Entity Recognition (NER)** to extract which drug and which reaction
- **Severity classification** — minor/moderate/severe
- **Drug-specific models** fine-tuned on condition-specific corpora (BioBERT, ClinicalBERT)
- **Active learning** — use model confidence to select most informative samples for annotation

### Connection to Prior Work
This project complements my medical imaging work by applying ML to a different modality of medical data — unstructured patient-reported text rather than imaging. Both address the challenge of detecting rare events (aneurysms, nodules, ADRs) in imbalanced datasets using domain-appropriate architectures and loss functions.

---
## References

1. Devlin, J., et al. (2018). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. *NAACL 2019.*
2. Sanh, V., et al. (2019). DistilBERT, a distilled version of BERT. *NeurIPS 2019 Workshop.*
3. Weissenbacher, D., et al. (2019). Overview of the Fourth Social Media Mining for Health (SMM4H) Shared Tasks at ACL 2019. *ACL 2019.*
4. Nikfarjam, A., et al. (2015). Pharmacovigilance from social media. *Journal of the American Medical Informatics Association.*
